# Stimuli from Cusimano, Hewitt & McDermott (2024), Appendix C

Milestone (a) of the design: the mistuned harmonic (C.6), ABA bistability (C.10) and
asynchronous onsets (C.7), rebuilt with sonore from the paper's text. The details the paper
leaves open, and what we assumed, are in each module's docstring.

Playback is normalized by the browser player, so the relative levels between stimuli are not
preserved here. The levels themselves are checked by the tests in `tests/stimuli/`.

In [ ]:
# On Colab, install the package with its stimuli extra. Locally, `pip install -e ".[stimuli]"` instead.
import importlib.util

if importlib.util.find_spec("sonore_inference") is None:
    %pip install -q "sonore-inference[stimuli] @ git+https://github.com/choyun1/sonore-inference"

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import mlab
import numpy as np
from IPython.display import Audio, display

from sonore_inference import stimuli


def show(sound, title, max_freq=5000):
    data = sound.data[:, 0]
    t = np.arange(len(data)) / sound.fs
    fig, (wave_ax, spec_ax) = plt.subplots(2, 1, figsize=(9, 4.5), sharex=True)
    wave_ax.plot(t, data, linewidth=0.5)
    wave_ax.set(title=title, ylabel="amplitude")
    power, freqs, times = mlab.specgram(data, NFFT=256, Fs=sound.fs, noverlap=224)
    power_db = 10 * np.log10(power + 1e-30)
    spec_ax.pcolormesh(times, freqs, power_db, vmin=power_db.max() - 70, cmap="magma")  # 70 dB range
    spec_ax.set(ylim=(0, max_freq), xlabel="time (s)", ylabel="frequency (Hz)")
    plt.show()
    display(Audio(data, rate=sound.fs))

## Mistuned harmonic (C.6)

200 Hz complex tone, harmonics 1-12 at 60 dB each, with the 3rd harmonic in tune and then
mistuned upward by 50% of f0.

In [ ]:
for percent in (0, 50):
    sound = stimuli.mistuned_harmonic_stimulus(200.0, 3, percent)
    show(sound, f"f0 200 Hz, harmonic 3 mistuned by {percent}% of f0", max_freq=3000)

## ABA bistability (C.10)

A = 1000 Hz, B 1 and 12 semitones higher, 100 ms onset-to-onset interval, four ABA triplets,
padded to 3.1 s.

In [ ]:
for semitones in (1, 12):
    sound = stimuli.bistability_stimulus(0.100, semitones)
    show(sound, f"B {semitones} semitones above A, 100 ms onset-to-onset", max_freq=3000)

## Asynchronous onsets (C.7)

The vowel with F1 = 438 Hz in each of the four continua: alone, with a synchronous 500 Hz
tone, and with the tone starting 32 or 240 ms early.

In [ ]:
for continuum in stimuli.asynchronous_onsets.CONTINUA:
    sound = stimuli.asynchronous_onsets_stimulus(continuum, 438.0)
    show(sound, f"{continuum}, F1 438 Hz")